# 第 2 周第 3 天练习 —— 樟宜免税店助手（iShopChangi + Gradio Chat）

## 练习目标（理念）

做一个面向旅客的 **免税店导购聊天机器人**：

1. 抓取 iShopChangi 若干分类页，拼成 `ishop_context` 作为回答依据
2. 用带 **history** 的 Gradio `ChatInterface` 做多轮对话
3. 对特定关键词（如 `iphone`）动态追加 system 提示，礼貌说明不卖并给替代建议

注意：原作者已写明——iShopChangi 是 **JS 渲染的 SPA**，`requests`+BeautifulSoup 往往只能拿到静态壳，商品列表可能很少；练习重点是「RAG 式塞上下文 + 多轮聊天」骨架，不是完美爬虫。

## 和本课 Week 2 的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 网站内容接地（grounding） | `ishop_context` 写进 system_message |
| 多轮 history | Gradio history → OpenAI messages 列表 |
| 条件式 system 改写 | 检测到 iphone 就追加说明 |
| 流式 Chat UI | `stream=True` + `yield` 累积回复 |

## 怎么跑

1. 本机 Ollama 已运行且有 `llama3.2`
2. 运行抓取格（需能访问 ishopchangi.com）
3. 最后一格 `launch` 聊天界面；可用 examples 快速试问


In [1]:
# ========== 导入：抓取、解析 HTML、调本地模型、搭 Gradio 聊天界面 ==========

# 导入 requests：HTTP GET 免税店页面
import requests
# 从 bs4 导入 BeautifulSoup：解析并清洗 HTML
from bs4 import BeautifulSoup
# 从 openai 导入 OpenAI：经 /v1 兼容接口调用本机 Ollama
from openai import OpenAI
# 导入 gradio：用 ChatInterface 做多轮导购对话
import gradio as gr


In [2]:
# ========== 连接 Ollama：OpenAI 兼容客户端指向本机 /v1 ==========

# 连接 Ollama——无需 API Key（api_key 占位即可）
ollama = OpenAI(
    api_key="ollama",
    base_url="http://localhost:11434/v1"
)

# 本地模型名常量；须与本机已安装模型一致
MODEL = "llama3.2"


In [3]:
# ========== 抓取 iShopChangi：多分类页拼成 ishop_context（注意 SPA 限制）==========

# 浏览器 User-Agent：降低被站点当成异常爬虫直接拒绝的概率
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}

# iShopChangi 是 JS 渲染的 SPA——requests+BeautifulSoup 只能抓到
# 静态 HTML 壳（meta、SEO、导航文案），而不是
# 完整渲染后的商品列表。
# 分类 URL 字典：key 用于日志，value 勿改（改 URL 等于换数据源）
ISHOP_PAGES = {
    "home":               "https://www.ishopchangi.com/en/home?cmode=tr",
    "liquor_tobacco":     "https://www.ishopchangi.com/en/category/liquor-and-tobacco",
    "perfumes_cosmetics": "https://www.ishopchangi.com/en/category/perfumes-and-cosmetics",
    "confectionery":      "https://www.ishopchangi.com/en/category/confectionery-and-food-beverage",
    "electronics":        "https://www.ishopchangi.com/en/category/electronics",
    "fashion":            "https://www.ishopchangi.com/en/category/fashion-watches-and-jewellery",
}

def fetch_page(url, max_chars=1500):
    # 抓取单页并截断，控制塞进 prompt 的体积
    try:
        resp = requests.get(url, headers=HEADERS, timeout=10)
        soup = BeautifulSoup(resp.content, "html.parser")
        # 标题缺失时用 "No title"（英文占位保持原样）
        title = soup.title.string.strip() if soup.title else "No title"
        if soup.body:
            # 去掉脚本/样式/媒体/导航等，尽量留可读文本
            for tag in soup.body(["script", "style", "img", "input", "nav", "footer"]):
                tag.decompose()
            text = soup.body.get_text(separator="\n", strip=True)
        else:
            text = ""
        return f"Page: {title}\nURL: {url}\n\n{text}"[:max_chars]
    except Exception as e:
        # 单页失败不中断整体；返回错误说明字符串
        return f"Could not fetch {url}: {e}"

def fetch_ishop_context():
    # 逐页抓取，用分隔线拼成一大段上下文
    parts = []
    for name, url in ISHOP_PAGES.items():
        print(f"Fetching {name} page...")
        parts.append(fetch_page(url))
    return "\n\n---\n\n".join(parts)

# 启动时加载网站内容到全局 ishop_context，供 system_message 引用
print("Loading iShopChangi content...")
ishop_context = fetch_ishop_context()
print(f"Done! Fetched {len(ishop_context)} characters of content.")


Loading iShopChangi content...
Fetching home page...
Fetching liquor_tobacco page...
Fetching perfumes_cosmetics page...
Fetching confectionery page...
Fetching electronics page...
Fetching fashion page...
Done! Fetched 606 characters of content.


In [4]:
# ========== system_message：导购人设 + 把 ishop_context 嵌进 system（英文指令与站点内容保持原样）==========

# 用 f-string 把抓到的网站文本写进 system，让模型回答「有据可依」
system_message = f"""You are a friendly and knowledgeable duty-free shopping assistant for iShopChangi — Changi Airport's official online duty-free store.
Help travellers discover products, understand duty-free allowances, find promotions, and navigate the shopping categories.
You remember the conversation history so you can follow up naturally on previous messages.
If a shopper mentions a specific category (e.g. whisky, perfume, snacks), recommend exploring that section and highlight any deals.
Be concise, warm, and practical. Respond in markdown without code blocks.

Here is the current iShopChangi website content to ground your answers:

{ishop_context}
"""


In [ ]:
# ========== chat：把 Gradio history 转成 messages，必要时追加 iPhone 说明，再流式 yield ==========

def chat(message, history):
    # Rebuild the message list from gradio history + new user message
    # 默认先用完整 system_message
    relevant_system_message = system_message
    # 若用户消息里提到 iphone（大小写不敏感），追加「本店不卖 iPhone」的英文提示（字符串勿译）
    if "iphone" in message.lower():
        relevant_system_message += (
            " If the customer asks about iPhone, let them know that iShopChangi does not sell iPhones. "
            "Politely suggest alternatives such as fragrances, premium spirits, cosmetics, confectionery, "
            "or fashion and jewellery available in the store."
        )

    # 先放 system，再展开 history 里已有的 role/content，最后追加本轮 user
    messages = [{"role": "system", "content": relevant_system_message}]
    messages += [{"role": h["role"], "content": h["content"]} for h in history]
    messages.append({"role": "user", "content": message})

    # 流式调用本地 MODEL
    stream = ollama.chat.completions.create(
        model=MODEL,
        messages=messages,
        stream=True
    )

    # 累积 response，每次 yield 给 ChatInterface 做打字机效果
    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ""
        yield response


In [6]:
# ========== 启动 Gradio ChatInterface：多轮导购 UI（title/description/examples 英文保持原样）==========

gr.ChatInterface(
    fn=chat,
    # type="messages"：history 元素是 {role, content} 字典，与上面 chat() 的解析方式一致
    type="messages",
    title="iShopChangi Duty-Free Assistant",
    description="Chat with your personal duty-free shopping guide for Changi Airport. Ask about products, brands, categories, allowances, or promotions.",
    examples=[
        "What product categories are available at iShopChangi?",
        "What whisky brands can I find in the liquor section?",
        "What perfumes and cosmetics brands are available?",
        "What electronics can I buy duty-free at Changi?",
        "What local Singapore snacks can I buy as gifts?",
        "What duty-free allowances should I know about?",
    ]
).launch(inbrowser=True)


* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
